In [13]:
from torch.nn.utils.rnn import pad_sequence


def collate_fn(batch):
    i3d, mp, txt = zip(*batch)

    fixed_i3d = []
    fixed_mp = []
    fixed_txt = []

    for x in i3d:
        # already tensor from dataset
        x = x.float()

        # remove useless dims
        x = x.squeeze()

        # ensure shape = (T, 1024)
        if x.dim() == 2:
            if x.shape[0] == 1024 and x.shape[1] != 1024:
                x = x.transpose(0, 1)

        fixed_i3d.append(x)

    for x in mp:
        x = x.float().squeeze()

        # ensure 2D => (T, 99)
        if x.dim() > 2:
            x = x.view(x.shape[0], -1)

        fixed_mp.append(x)

    for x in txt:
        fixed_txt.append(x.long())

    # pad variable-length sequences
    i3d = pad_sequence(fixed_i3d, batch_first=True)          # (B, Ti, 1024)
    mp  = pad_sequence(fixed_mp, batch_first=True)           # (B, Tm, 99)
    txt = pad_sequence(fixed_txt, batch_first=True, padding_value=0)

    return i3d, mp, txt


In [17]:
import pandas as pd
import os
from src.data.how2sign import How2SignDataset
from src.utils.tokenizer import Tokenizer
from torch.utils.data import DataLoader, Subset
import random

os.chdir("D:/SignDetection")
TRAIN_CSV = 'datasets/raw/how2sign/tsv_files_how2sign/tsv_files_how2sign/cvpr23.fairseq.i3d.train.how2sign.tsv'
base_i3d_train = os.path.abspath("datasets/raw/how2sign/i3d_features_how2sign/i3d_features_how2sign/train")
base_mp_train = os.path.abspath("datasets/raw/how2sign/mediapipe_features_how2sign/mediapipe_features/train")

BATCH_SIZE = 8
train_df = pd.read_csv(os.path.abspath(TRAIN_CSV), sep="\t")

tokenizer = Tokenizer()
tokenizer.build_vocab(train_df["translation"].tolist())

train_ds = How2SignDataset(os.path.abspath(TRAIN_CSV), tokenizer, base_mp=base_mp_train, base_i3d=base_i3d_train)

# use 10% ds
n = len(train_ds)
subset_size = int(0.01 * n)

indices = random.sample(range(n), subset_size)

train_ds = Subset(train_ds, indices)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

for i3d, mp, text in train_loader:
    print(f"i3d: {i3d.shape}, mp: {mp.shape}, text: {text.shape}")


i3d: torch.Size([8, 549, 1024]), mp: torch.Size([8, 539, 99]), text: torch.Size([8, 38])
i3d: torch.Size([8, 565, 1024]), mp: torch.Size([8, 556, 99]), text: torch.Size([8, 56])
i3d: torch.Size([8, 571, 1024]), mp: torch.Size([8, 483, 99]), text: torch.Size([8, 61])
i3d: torch.Size([8, 341, 1024]), mp: torch.Size([8, 367, 99]), text: torch.Size([8, 31])
i3d: torch.Size([8, 788, 1024]), mp: torch.Size([8, 816, 99]), text: torch.Size([8, 65])
i3d: torch.Size([8, 264, 1024]), mp: torch.Size([8, 522, 99]), text: torch.Size([8, 37])
i3d: torch.Size([8, 279, 1024]), mp: torch.Size([8, 384, 99]), text: torch.Size([8, 35])
i3d: torch.Size([8, 218, 1024]), mp: torch.Size([8, 232, 99]), text: torch.Size([8, 33])
i3d: torch.Size([8, 264, 1024]), mp: torch.Size([8, 414, 99]), text: torch.Size([8, 33])
i3d: torch.Size([8, 757, 1024]), mp: torch.Size([8, 742, 99]), text: torch.Size([8, 66])
i3d: torch.Size([8, 383, 1024]), mp: torch.Size([8, 291, 99]), text: torch.Size([8, 38])
i3d: torch.Size([8, 5